## Mode Selection

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # ============================================================
    # OPTIMIZED LoRA CONFIG (v8) — targeting 92%+ accuracy
    # ============================================================
    # Rank 32 is the competition maximum.
    # RSLoRA uses alpha/sqrt(r) scaling — already regularizes,
    # so dropout=0.0 avoids double-regularization penalty.
    # lm_head REMOVED: prevents format drift on \\boxed{} output.
    # ============================================================
    LORA_RANK = 32
    LORA_ALPHA = 64          # 2x rank — strong adapter influence
    LORA_DROPOUT = 0.0       # RSLoRA already regularizes; dropout hurts

    target_modules = [
        # Attention projections (core reasoning)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj",

        # MLP / MoE (feed-forward reasoning capacity)
        "gate_proj", "up_proj", "down_proj",

        # Mamba-specific (critical for Nemotron's hybrid architecture)
        "x_proj", "dt_proj",

        # NOTE: lm_head intentionally EXCLUDED to prevent
        # output distribution drift on \\boxed{} formatting
    ]

    print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
    print(f"  Rank={LORA_RANK}, Alpha={LORA_ALPHA}, Dropout={LORA_DROPOUT}")
    print(f"  Target modules: {target_modules}")
    print(f"  RSLoRA=True, lm_head=EXCLUDED")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")


## Mode A: Train on Kaggle

In [ ]:
# if TRAIN_ON_KAGGLE:
#     import pandas as pd
#     import random
#     import gc, time
#     from datasets import Dataset as HFDataset
#     from trl import SFTTrainer, SFTConfig

#     SEED = 42
#     PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'


#     DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
#     df = pd.read_csv(DATASET_PATH)
#     print(f"Full dataset: {len(df)} rows")


#     train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
#     print(f"Full dataset: {len(df)} rows")


#     import re
#     import math
#     from collections import defaultdict
#     from torch.utils.data import DataLoader, Sampler
#     records = []
#     record_types = []
#     for _, row in train_df.iterrows():
#         prompt = str(row["prompt"])
#         answer = str(row["answer"])
#         cot = str(row["generated_cot"])
#         if not cot or cot == "nan" or len(cot.strip()) < 5:
#             continue
#         cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
#         user_content = prompt + PROMPT_SUFFIX
#         assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
#         records.append({"messages": [
#             {"role": "user", "content": user_content},
#             {"role": "assistant", "content": assistant_content},
#         ]})
#         record_types.append(str(row["type"]))
#     dataset = HFDataset.from_list(records)
#     print(f"SFT records: {len(records)}")

#     def formatting_prompts_func(example):
#         messages = example["messages"]
#         if messages and isinstance(messages[0], dict):
#             conversations = [messages]
#         else:
#             conversations = messages

#         texts = []
#         for conversation in conversations:
#             try:
#                 text = tokenizer.apply_chat_template(
#                     conversation,
#                     tokenize=False,
#                     add_generation_prompt=False,
#                     enable_thinking=True,
#                 )
#             except TypeError:
#                 text = tokenizer.apply_chat_template(
#                     conversation,
#                     tokenize=False,
#                     add_generation_prompt=False,
#                 )
#             texts.append(text)
#         return texts


#     training_args = SFTConfig(
#         output_dir="/kaggle/working/sft_output",
#         num_train_epochs=1,
#         per_device_train_batch_size=2,
#         gradient_accumulation_steps=4,
#         learning_rate=8e-5,
#         lr_scheduler_type="cosine",
#         warmup_ratio=0.05,

#         max_length=8192,
#         optim="paged_adamw_8bit",

#         adam_beta1=0.9,
#         adam_beta2=0.95,
#         adam_epsilon=1e-8,

#         weight_decay=0.01,
#         max_grad_norm=1.0,

#         # LOGGING
#         logging_steps=10,
#         save_strategy="no",
#         bf16=True,
#         gradient_checkpointing=True,
#         gradient_checkpointing_kwargs={"use_reentrant": False},
#         dataloader_num_workers=2,
#         remove_unused_columns=False,
#         seed=SEED,
#         report_to="none",
#         packing=False,
#     )

#     def build_stratified_index_order(labels, batch_size, seed):
#         """Approximate nemotron-master's stratified batching over effective batches."""
#         by_label = defaultdict(list)
#         for idx, label in enumerate(labels):
#             by_label[label].append(idx)

#         rng = random.Random(seed)
#         for idx_list in by_label.values():
#             rng.shuffle(idx_list)

#         n_batches = max(1, math.ceil(len(labels) / batch_size))
#         batches = [[] for _ in range(n_batches)]
#         batch_order = list(range(n_batches))
#         rng.shuffle(batch_order)

#         assigned = 0
#         for label in sorted(by_label.keys()):
#             for idx in by_label[label]:
#                 batches[batch_order[assigned % n_batches]].append(idx)
#                 assigned += 1

#         order = [idx for batch in batches for idx in batch]
#         if len(order) != len(labels):
#             raise ValueError("Stratified order size mismatch")
#         return order

#     class PrecomputedOrderSampler(Sampler):
#         def __init__(self, order):
#             self.order = list(order)

#         def __iter__(self):
#             return iter(self.order)

#         def __len__(self):
#             return len(self.order)

#     class StratifiedSFTTrainer(SFTTrainer):
#         def __init__(self, *args, stratified_order=None, **kwargs):
#             super().__init__(*args, **kwargs)
#             self.stratified_order = stratified_order

#         def get_train_dataloader(self):
#             if self.train_dataset is None:
#                 raise ValueError("Trainer requires a train_dataset.")
#             if self.stratified_order is None:
#                 return super().get_train_dataloader()
#             if len(self.stratified_order) != len(self.train_dataset):
#                 raise ValueError("Stratified order length does not match train dataset")

#             dataloader_kwargs = {
#                 "batch_size": self.args.per_device_train_batch_size,
#                 "sampler": PrecomputedOrderSampler(self.stratified_order),
#                 "collate_fn": self.data_collator,
#                 "num_workers": self.args.dataloader_num_workers,
#                 "pin_memory": self.args.dataloader_pin_memory,
#                 "persistent_workers": self.args.dataloader_persistent_workers,
#                 "drop_last": self.args.dataloader_drop_last,
#             }
#             if self.args.dataloader_num_workers > 0:
#                 dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor

#             return DataLoader(self.train_dataset, **dataloader_kwargs)

#     effective_batch_size = max(
#         1,
#         training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
#     )
#     stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
#     print(f"Approx stratified effective batch size: {effective_batch_size}")
#     print("Stratified batching by type:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

#     trainer = StratifiedSFTTrainer(
#         model=model,
#         args=training_args,
#         train_dataset=dataset,
#         processing_class=tokenizer,
#         formatting_func=formatting_prompts_func,
#         stratified_order=stratified_order,
#     )

#     print("Starting SFT training...")
#     t0 = time.time()
#     trainer.train()
#     elapsed = time.time() - t0
#     print(f"Training done in {elapsed/60:.1f} min")


#     ADAPTER_DIR = "/kaggle/working/sft_adapter"
#     model.save_pretrained(ADAPTER_DIR)
#     tokenizer.save_pretrained(ADAPTER_DIR)
#     print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
if TRAIN_ON_KAGGLE:
    # ============================================================
    # MEMORY OPTIMIZATIONS (must be set before any CUDA calls)
    # ============================================================
    import os
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
    
    import pandas as pd
    import random
    import gc
    import time
    import torch
    import re
    import math
    from collections import defaultdict
    from torch.utils.data import DataLoader, Sampler
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    SEED = 42
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")

    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Shuffled dataset: {len(train_df)} rows")

    # -------------------------------
    # Build dataset with messages
    # FIX: Added opening <think> tag that was missing in v7-5
    # -------------------------------
    records = []
    record_types = []
    skipped = 0
    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            skipped += 1
            continue
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        user_content = prompt + PROMPT_SUFFIX
        # CRITICAL FIX: v7-5 was missing the opening <think> tag!
        # This caused format misalignment between training and inference.
        assistant_content = "<think>\n" + cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
        records.append({
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content},
            ]
        })
        record_types.append(str(row["type"]))
    
    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)} (skipped {skipped} invalid CoT)")

    # -------------------------------
    # Formatting function for chat template
    # -------------------------------
    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages
        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=True,
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            texts.append(text)
        return texts

    # ============================================================
    # OPTIMIZED TRAINING ARGS (v8)
    # ============================================================
    # Key changes from v7-5:
    #   - LR: 8e-5 → 2e-4  (optimal for LoRA fine-tuning)
    #   - Epochs: 2 → 3     (more exposure to reasoning patterns)
    #   - Warmup: 0 → 0.1   (10% warmup prevents early instability)
    #   - NEFTune: 5.0       (noisy embeddings = +2-5% generalization)
    #   - Weight decay: 0.01 → 0.005 (preserve base model knowledge)
    #   - Grad norm: 1.0 → 0.5 (tighter clipping for stability)
    #   - Beta2: 0.95 → 0.999 (standard Adam, less aggressive)
    #   - Packing: True      (eliminates padding waste)
    #   - use_reentrant: False (reliable gradient checkpointing)
    # ============================================================
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=3,                        # v7: 2 → v8: 3
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,             # effective batch = 8

        # ── Learning rate ──
        learning_rate=2e-4,                        # v7: 8e-5 → v8: 2e-4
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,                          # v7: MISSING → v8: 10%

        max_length=8192,
        optim="paged_adamw_8bit",

        # ── Optimizer params ──
        adam_beta1=0.9,
        adam_beta2=0.999,                           # v7: 0.95 → v8: 0.999
        adam_epsilon=1e-8,

        # ── Regularization ──
        weight_decay=0.005,                        # v7: 0.01 → v8: 0.005
        max_grad_norm=0.5,                         # v7: 1.0 → v8: 0.5
        neftune_noise_alpha=5.0,                   # v7: NONE → v8: 5.0

        # ── Logging & saving ──
        logging_steps=10,
        save_strategy="no",
        bf16=True,

        # ── Memory optimization ──
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},  # v7: True → v8: False

        # ── Data handling ──
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=True,                              # v7: False → v8: True
        dataset_num_proc=4,
    )

    # Print configuration summary
    print("\n" + "="*60)
    print("  OPTIMIZED TRAINING CONFIG (v8)")
    print("="*60)
    print(f"  LR:           {training_args.learning_rate}")
    print(f"  Epochs:       {training_args.num_train_epochs}")
    print(f"  Warmup:       {training_args.warmup_ratio}")
    print(f"  NEFTune:      {training_args.neftune_noise_alpha}")
    print(f"  Batch:        {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
    print(f"  Weight Decay: {training_args.weight_decay}")
    print(f"  Grad Norm:    {training_args.max_grad_norm}")
    print(f"  Packing:      {training_args.packing}")
    print("="*60 + "\n")

    # -------------------------------
    # Stratified batching helpers (unchanged from v7-5)
    # -------------------------------
    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)
        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)
        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)
        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1
        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)
        def __iter__(self):
            return iter(self.order)
        def __len__(self):
            return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order
        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            if len(self.stratified_order) != len(self.train_dataset):
                raise ValueError("Stratified order length does not match train dataset")
            dataloader_kwargs = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
            return DataLoader(self.train_dataset, **dataloader_kwargs)

    effective_batch_size = max(
        1,
        training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    )
    stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
    print(f"Approx stratified effective batch size: {effective_batch_size}")
    print("Stratified batching by type:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

    trainer = StratifiedSFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
    )

    # -------------------------------
    # Clear cache before training
    # -------------------------------
    torch.cuda.empty_cache()
    gc.collect()

    print("Starting SFT training (v8 optimized)...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed/60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")


## Phase 2: GRPO (Reinforcement Learning with Verifiable Rewards)

After SFT, we apply **GRPO** (Group Relative Policy Optimization) using the competition's
training data with **verifiable reward functions**. The model generates multiple completions
per prompt, and correct answers are reinforced.

**Why GRPO + RLVR?**
- RLVR = reward paradigm (verifiable answers → binary correctness signal)
- GRPO = optimization algorithm (no critic model needed → fits in GPU memory)
- All competition tasks are 100% verifiable (bit manipulation, Roman numerals, unit conversion, etc.)

In [ ]:
if TRAIN_ON_KAGGLE:
    import re
    import gc
    import torch
    import time

    # ============================================================
    # REWARD FUNCTIONS for GRPO (RLVR approach)
    # ============================================================
    # These are deterministic verifiers — no learned reward model needed.
    # Each function scores model completions against ground truth.
    # ============================================================

    def extract_boxed_answer(text: str) -> str:
        """Extract the content inside \\boxed{...} with brace-balanced parsing."""
        idx = text.rfind("\\boxed{")
        if idx == -1:
            # Fallback: try to find the last line as the answer
            lines = text.strip().split("\n")
            return lines[-1].strip() if lines else ""
        depth, start = 1, idx + 7
        for i in range(start, len(text)):
            if text[i] == '{': depth += 1
            elif text[i] == '}': depth -= 1
            if depth == 0:
                return text[start:i].strip()
        return text[start:].strip()

    def is_numeric(s: str) -> bool:
        """Check if a string represents a number."""
        try:
            float(s.replace(",", ""))
            return True
        except (ValueError, AttributeError):
            return False

    def accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
        """
        CORE REWARD: Exact match or numerical tolerance (±0.02).
        This is the primary RLVR signal — binary correctness.
        """
        rewards = []
        for completion, truth in zip(completions, answer):
            response = completion[0]["content"] if isinstance(completion, list) else str(completion)
            predicted = extract_boxed_answer(response)
            truth_str = str(truth).strip()

            # Exact string match (handles bit manipulation, Roman numerals, encryption)
            if predicted == truth_str:
                rewards.append(2.0)  # Strong positive signal
                continue

            # Case-insensitive match (for text answers)
            if predicted.lower().strip() == truth_str.lower().strip():
                rewards.append(2.0)
                continue

            # Numerical tolerance (±0.02) for unit conversion, gravity problems
            if is_numeric(predicted) and is_numeric(truth_str):
                try:
                    pred_val = float(predicted.replace(",", ""))
                    true_val = float(truth_str.replace(",", ""))
                    if abs(pred_val - true_val) < 0.02:
                        rewards.append(2.0)
                        continue
                    # Partial credit for close answers (within 5%)
                    if true_val != 0 and abs(pred_val - true_val) / abs(true_val) < 0.05:
                        rewards.append(0.5)
                        continue
                except:
                    pass

            rewards.append(0.0)  # Wrong answer
        return rewards

    def format_reward_func(prompts, completions, **kwargs) -> list[float]:
        """
        FORMAT REWARD: Encourages proper <think>...</think>\\boxed{} structure.
        """
        rewards = []
        for completion in completions:
            response = completion[0]["content"] if isinstance(completion, list) else str(completion)
            score = 0.0

            # Check for <think> tag
            if "<think>" in response:
                score += 0.2
            # Check for </think> tag
            if "</think>" in response:
                score += 0.2
            # Check for \boxed{}
            if "\\boxed{" in response:
                score += 0.4
            # Bonus: proper ordering (think before boxed)
            think_end = response.rfind("</think>")
            boxed_start = response.rfind("\\boxed{")
            if think_end > 0 and boxed_start > think_end:
                score += 0.2

            rewards.append(score)
        return rewards

    def reasoning_length_reward_func(prompts, completions, **kwargs) -> list[float]:
        """
        LENGTH REWARD: Penalizes too-short or too-long reasoning.
        Sweet spot: 100-2000 chars of reasoning.
        """
        rewards = []
        for completion in completions:
            response = completion[0]["content"] if isinstance(completion, list) else str(completion)
            # Extract reasoning portion (between <think> and </think>)
            think_match = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
            if think_match:
                reasoning = think_match.group(1)
                length = len(reasoning)
                if 100 <= length <= 2000:
                    rewards.append(0.3)   # Goldilocks zone
                elif 50 <= length < 100 or 2000 < length <= 4000:
                    rewards.append(0.1)   # Acceptable
                else:
                    rewards.append(-0.1)  # Too short or too long
            else:
                rewards.append(-0.2)  # No reasoning at all
        return rewards

    print("GRPO reward functions defined:")
    print("  1. accuracy_reward_func   (weight: primary, scores 0/0.5/2.0)")
    print("  2. format_reward_func     (weight: format, scores 0.0-1.0)")
    print("  3. reasoning_length_reward (weight: length, scores -0.2 to 0.3)")
else:
    print("USE_PRETRAINED=1: skipping GRPO reward function definitions.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import pandas as pd
    from datasets import Dataset as HFDataset

    # ============================================================
    # GRPO DATASET: Use competition train.csv directly
    # ============================================================
    # GRPO needs: prompt (the question) + answer (ground truth for reward)
    # The model generates its OWN reasoning — no CoT labels needed.
    # ============================================================

    PROMPT_SUFFIX_GRPO = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # Load competition training data
    comp_df = pd.read_csv("/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv")
    print(f"Competition train set: {len(comp_df)} problems")

    # Format for GRPOTrainer: needs 'prompt' column as list of messages
    grpo_records = []
    for _, row in comp_df.iterrows():
        prompt_text = str(row["prompt"]) + PROMPT_SUFFIX_GRPO
        grpo_records.append({
            "prompt": [
                {"role": "user", "content": prompt_text},
            ],
            "answer": str(row["answer"]),
        })

    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f"GRPO dataset prepared: {len(grpo_records)} prompts")
    print(f"Sample prompt (first 200 chars): {grpo_records[0]['prompt'][0]['content'][:200]}...")
    print(f"Sample answer: {grpo_records[0]['answer']}")
else:
    print("USE_PRETRAINED=1: skipping GRPO dataset preparation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    from trl import GRPOTrainer, GRPOConfig

    # ============================================================
    # GRPO TRAINING CONFIGURATION
    # ============================================================
    # Memory-optimized for Kaggle GPU (Blackwell/A100).
    # Uses the SFT adapter as starting point (already loaded in model).
    # GRPO generates `num_generations` completions per prompt,
    # scores them with reward functions, and reinforces the best ones.
    # ============================================================

    # Clear memory from SFT phase
    if 'trainer' in dir():
        del trainer
    if 'dataset' in dir():
        del dataset
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated")

    grpo_args = GRPOConfig(
        output_dir="/kaggle/working/grpo_output",

        # ── Generation (exploration) ──
        num_generations=4,                          # Group size: 4 completions per prompt
        max_completion_length=2048,                 # Shorter than SFT to save VRAM
        temperature=0.7,                            # Diversity for exploration

        # ── Training ──
        max_steps=500,                              # ~500 steps is usually enough
        per_device_train_batch_size=1,              # Memory constraint
        gradient_accumulation_steps=8,              # Effective batch = 8
        learning_rate=5e-6,                         # 40x lower than SFT — gentle RL updates
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,

        # ── KL penalty ──
        beta=0.04,                                  # KL constraint: don't drift too far from SFT

        # ── Memory optimization ──
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",

        # ── Logging ──
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=42,
    )

    print("\n" + "="*60)
    print("  GRPO TRAINING CONFIG")
    print("="*60)
    print(f"  Generations:  {grpo_args.num_generations}")
    print(f"  Max tokens:   {grpo_args.max_completion_length}")
    print(f"  Temperature:  {grpo_args.temperature}")
    print(f"  Max steps:    {grpo_args.max_steps}")
    print(f"  LR:           {grpo_args.learning_rate}")
    print(f"  KL beta:      {grpo_args.beta}")
    print(f"  Batch:        {grpo_args.per_device_train_batch_size} x {grpo_args.gradient_accumulation_steps}")
    print("="*60 + "\n")

    # Initialize GRPO trainer with reward functions
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[
            accuracy_reward_func,        # Primary: correct answer → 2.0
            format_reward_func,           # Secondary: proper format → 0-1.0
            reasoning_length_reward_func, # Tertiary: good CoT length → -0.2 to 0.3
        ],
        args=grpo_args,
        train_dataset=grpo_dataset,
    )

    print("Starting GRPO training (Phase 2)...")
    t0 = time.time()
    grpo_trainer.train()
    elapsed = time.time() - t0
    print(f"GRPO training done in {elapsed/60:.1f} min")

    # Save the GRPO-refined adapter (overwrites SFT adapter)
    GRPO_ADAPTER_DIR = "/kaggle/working/sft_adapter"  # Same path — submission uses this
    model.save_pretrained(GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
    print(f"GRPO adapter saved to {GRPO_ADAPTER_DIR}")
    print("This adapter now has SFT + GRPO improvements.")

    # Cleanup
    del grpo_trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("USE_PRETRAINED=1: skipping GRPO training.")


## Mode B: Load Pre-trained LoRA（Temporarily unavailable）

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")
